In [90]:
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa

FOLDER_NAME = 'annotations'

FILE_NAMES = [
    '001_Adam',
    '002_Gianfranco',
    '003_Javier',
    '004_Marino',   
]

SHEET_NAMES = [
    'CSC',
    'MultiPICo',
]

ACCEPTABLE_LABELS = ['non-triggering', 'triggering']

In [91]:
label_to_index = {label: i for i, label in enumerate(ACCEPTABLE_LABELS)}
label_to_index

{'non-triggering': 0, 'triggering': 1}

In [92]:
# load dfs
dfs = {}
for file_name in FILE_NAMES:
    for sheet_name in SHEET_NAMES:
        df = pd.read_excel(f'{FOLDER_NAME}/{file_name}.xlsx', sheet_name=sheet_name)
        df.rename(columns={
            'Given this conversational context, is an ironic response likely or natural?': 'label'
        }, inplace=True)
        dfs[f'{file_name}_{sheet_name}'] = df

print(dfs.keys())
dfs['001_Adam_CSC'].head()

dict_keys(['001_Adam_CSC', '001_Adam_MultiPICo', '002_Gianfranco_CSC', '002_Gianfranco_MultiPICo', '003_Javier_CSC', '003_Javier_MultiPICo', '004_Marino_CSC', '004_Marino_MultiPICo'])


,unique_id,dialogue,label,comment
0,111_1036,"Turn 1:\n u1: Steve, you've been drinking a l...",triggering,NaN
1,253_2082,Turn 1:\n u1: Look at this and tell me what y...,triggering,NaN
2,231_2069,"Turn 1:\n u1: Come on, it happened like one t...",triggering,NaN
3,265_2014,Turn 1:\n u1: I'd really appreciate it. It's ...,non-triggering,NaN
4,222_2019,Turn 1:\n u1: I found this one last week! It ...,non-triggering,NaN


In [93]:
# remove invalid labels

ids_to_remove = set()
for key, df in dfs.items():
    invalid_ids = df.loc[~df['label'].isin(ACCEPTABLE_LABELS), 'unique_id']
    ids_to_remove.update(invalid_ids)

print(len(ids_to_remove))
print(ids_to_remove)

21
{'208_2036', '1817886739073302709_8931910327729403976', '248_2078', '8755515661243036834_15818415332632239449', '266_2054', '115_1004', '15377547143975358910_1711386745880565388', '7199594443675332653_13942175382689258523', '17565001203181101696_9217991978256712534', '12504922140387146689_17713436929031669011', '13867510771314669985_3692101571900816325', '209_2038', '17260434174864013858_11445425254358103606', '111_1036', '13351787454385947951_4930900043121476558', '8648986929715964531_6080813350745733225', '15735844402404738656_4292779336955660346', '109_1010', '11804440989245452213_8697572573754267076', '17684334533513677021_15921899341881086227', '10302135371128027311_12495562528796468539'}


In [94]:
# filter dfs
for key in dfs.keys():
    dfs[key] = dfs[key][~dfs[key]['unique_id'].isin(ids_to_remove)]
    print(key, len(dfs[key]))

001_Adam_CSC 93
001_Adam_MultiPICo 86
002_Gianfranco_CSC 93
002_Gianfranco_MultiPICo 86
003_Javier_CSC 93
003_Javier_MultiPICo 86
004_Marino_CSC 93
004_Marino_MultiPICo 86


In [95]:
# calcolo Fleiss
for sheet_name in SHEET_NAMES:
    print(f'\n=== {sheet_name} ===')

    # prendi solo annotatori di questo sheet
    sheet_dfs = [dfs[key] for key in dfs if sheet_name in key]

    # 🔹 intersezione degli ID (IMPORTANTISSIMO)
    common_ids = set(sheet_dfs[0]['unique_id'])
    for df in sheet_dfs[1:]:
        common_ids &= set(df['unique_id'])
    common_ids = sorted(common_ids)

    matrix = []

    for uid in common_ids:
        row = [0] * len(ACCEPTABLE_LABELS)

        for df in sheet_dfs:
            label = df.loc[df['unique_id'] == uid, 'label'].values[0]
            row[label_to_index[label]] += 1

        matrix.append(row)

    matrix = np.array(matrix)

    # 🔹 controllo: ogni riga deve sommare a 4 annotatori
    assert np.all(matrix.sum(axis=1) == len(sheet_dfs)), "Errore: numero annotatori non costante"

    kappa = fleiss_kappa(matrix)
    print('Fleiss kappa:', kappa)



=== CSC ===
Fleiss kappa: 0.3561711385061639

=== MultiPICo ===
Fleiss kappa: 0.3034348282585871


In [96]:
import pandas as pd

# lista delle label accettabili
ACCEPTABLE_LABELS = ['triggering', 'non-triggering']
label_to_index = {label: i for i, label in enumerate(ACCEPTABLE_LABELS)}

gold_labels = {}

for sheet_name in SHEET_NAMES:
    print(f'\n=== {sheet_name} ===')

    # prendi solo annotatori di questo sheet
    sheet_dfs = [dfs[key] for key in dfs if sheet_name in key]

    # intersezione degli ID
    common_ids = set(sheet_dfs[0]['unique_id'])
    for df in sheet_dfs[1:]:
        common_ids &= set(df['unique_id'])
    common_ids = sorted(common_ids)

    for uid in common_ids:
        label_counts = [0] * len(ACCEPTABLE_LABELS)

        for df in sheet_dfs:
            label = df.loc[df['unique_id'] == uid, 'label'].values[0]
            label_counts[label_to_index[label]] += 1

        # voto maggioritario (tie-break → 'triggering')
        if label_counts[label_to_index['non-triggering']] > label_counts[label_to_index['triggering']]:
            gold_labels[uid] = '0'
        else:
            gold_labels[uid] = '1'

# 🔹 crea DataFrame
gold_df = pd.DataFrame({
    'unique_id': list(gold_labels.keys()),
    'gold_label': list(gold_labels.values())
})

# 🔹 salva CSV
gold_df.to_csv('annotations/gold_labels.csv', index=False)
print("CSV gold_labels.csv creato correttamente!")


=== CSC ===

=== MultiPICo ===
CSV gold_labels.csv creato correttamente!
CSV gold_labels.csv creato correttamente!


In [99]:
import json
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import precision_score, recall_score, f1_score

# ── paths ───────────────────────────────────────────────────────────────
GOLD_CSV = 'annotations/gold_labels.csv'
CSC_JSON = '../datasets/CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered.json'
MULTIPICO_JSON = '../datasets/MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered.json'

# ── load gold labels ────────────────────────────────────────────────────
gold_df = pd.read_csv(GOLD_CSV)
gold_df['gold_label'] = gold_df['gold_label'].astype(int)

# ── helper: estrai (unique_id, label) dal JSON ─────────────────────────
def load_dataset_labels(json_path, label_field):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    rows = []
    for entry in data['entries']:
        uid = entry['unique_id']
        lbl = entry['annotations'].get(label_field)
        rows.append({'unique_id': uid, 'dataset_label': lbl})
    return pd.DataFrame(rows)

# ── carica dataset ─────────────────────────────────────────────────────
csc_df       = load_dataset_labels(CSC_JSON, 'sarcasm_appropriateness_label')
multipico_df = load_dataset_labels(MULTIPICO_JSON, 'irony_appropriateness_label')
dataset_df   = pd.concat([csc_df, multipico_df], ignore_index=True)

# ── merge con gold labels ──────────────────────────────────────────────
merged = gold_df.merge(dataset_df, on='unique_id', how='inner').dropna(subset=['dataset_label'])
merged['dataset_label'] = merged['dataset_label'].astype(int)

print(f"Entries after merge & dropna: {len(merged)}")
print(merged[['unique_id', 'gold_label', 'dataset_label']].head())

# ── funzioni di valutazione ──────────────────────────────────────────────
def accuracy(y_true, y_pred):
    return (y_true == y_pred).mean()

def print_metrics(label, y_true, y_pred, n):
    sp, spp = spearmanr(y_true, y_pred)
    pr, prp = pearsonr(y_true, y_pred)
    acc     = accuracy(y_true, y_pred)
    prec    = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec     = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1      = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    print(f"\n── {label} (n={n}) ──")
    print(f"  Accuracy       : {acc:.4f}")
    print(f"  Precision (1)  : {prec:.4f}")
    print(f"  Recall    (1)  : {rec:.4f}")
    print(f"  F1        (1)  : {f1:.4f}")
    print(f"  Spearman       : ρ = {sp:.4f},  p = {spp:.4e}")
    print(f"  Pearson        : r = {pr:.4f},  p = {prp:.4e}")

# ── metriche globali ─────────────────────────────────────────────────────
print_metrics(
    "Global (all datasets combined)",
    merged['gold_label'], merged['dataset_label'], len(merged)
)

# ── breakdown per dataset ──────────────────────────────────────────────
for ds_name, ds_json, field in [
    ('CSC',       CSC_JSON,       'sarcasm_appropriateness_label'),
    ('MultiPICo', MULTIPICO_JSON, 'irony_appropriateness_label'),
]:
    ds_df = load_dataset_labels(ds_json, field)
    m = gold_df.merge(ds_df, on='unique_id', how='inner').dropna(subset=['dataset_label'])
    m['dataset_label'] = m['dataset_label'].astype(int)

    if len(m) < 2:
        print(f"\n{ds_name}: not enough overlapping entries ({len(m)})")
        continue

    print_metrics(ds_name, m['gold_label'], m['dataset_label'], len(m))

Entries after merge & dropna: 179
  unique_id  gold_label  dataset_label
0  101_1015           1              1
1  103_1014           0              1
2  103_1019           1              1
3  103_1031           1              1
4  103_1034           1              1

── Global (all datasets combined) (n=179) ──
  Accuracy       : 0.6201
  Precision (1)  : 0.5682
  Recall    (1)  : 0.6250
  F1        (1)  : 0.5952
  Spearman       : ρ = 0.2398,  p = 1.2224e-03
  Pearson        : r = 0.2398,  p = 1.2224e-03

── CSC (n=93) ──
  Accuracy       : 0.6129
  Precision (1)  : 0.6087
  Recall    (1)  : 0.6087
  F1        (1)  : 0.6087
  Spearman       : ρ = 0.2257,  p = 2.9596e-02
  Pearson        : r = 0.2257,  p = 2.9596e-02

── MultiPICo (n=86) ──
  Accuracy       : 0.6279
  Precision (1)  : 0.5238
  Recall    (1)  : 0.6471
  F1        (1)  : 0.5789
  Spearman       : ρ = 0.2567,  p = 1.7041e-02
  Pearson        : r = 0.2567,  p = 1.7041e-02

── CSC (n=93) ──
  Accuracy       : 0.6129
  Prec